# Beyond Prediction: Comparing Manual, Pipeline-Based, and Insight-Driven ML Approaches for House Prices

This project compares three different approaches to building a house price prediction model: a manual workflow, a purely technical pipeline, and a combined approach that integrates data-driven insights into a reproducible pipeline. The goal is to evaluate how domain understanding and technical structure interact and how they influence model performance.The first part of the analysis focuses on a manual, step-by-step workflow, while the final sections introduce pipeline-based implementations to enable a structured comparison of different worklflow approaches.

Structure of this analysis:

- 1. Data Acquisition
- 2. Data Quality Assessment
- 3. Missing Value Analysis and Imputation
    - 1. Structural Missing Value Imputation
    - 2. Statistical Missing Value Imputation
    - 3. Dropped Features
    - 4. Final Missing Value Check
- 4. Feature Engineering
    - 1. Encoding categorical Features
    - 2. Constructing Features 
- 5. Feature Selection
- 6. Train-Test Split
- 7. Model Training and Baseline Evaluation
- 8. Model Improvement
- 9. Model Selection
- 10. Excursus: Explainability - extracting Decision Rules
- 11. Pipeline based Re-Implementation and Comparison
    - 1. General Best-Practice Pipeline
    - 2. Insight-Driven Pipeline
    - 3. Comparison of Workflow Approaches
- 12. Conclusion

## 1. Data Acquisition
Dataset is obtained from Kaggle, it was scraped from publicly available results posted every week from Domain.com.au. It is already cleaned. As a first step, we load the data and take a first look at it.

In [1]:
# loading data into a pandas DataFrame and checking the first few rows, transposed for better readability
import pandas as pd
df = pd.read_csv("train_housing_prices.csv")
df.head().T


,0,1,2,3,4
Id,1,2,3,4,5
MSSubClass,60,20,60,70,60
MSZoning,RL,RL,RL,RL,RL
LotFrontage,65.0,80.0,68.0,60.0,84.0
LotArea,8450,9600,11250,9550,14260
...,...,...,...,...,...
MoSold,2,5,9,2,12
YrSold,2008,2007,2008,2006,2008
SaleType,WD,WD,WD,WD,WD
SaleCondition,Normal,Normal,Normal,Abnorml,Normal


## 2. Data Quality Assessment

Since the dataset was made available from Kaggle for the exact purpose of building first ML workflows, there should be no big surprises.
We start with a first overview, basic informarion about the data and some statistics

In [2]:
# checking info about the dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

The data types appear to be consistent with the corresponding feature names. The only exception is "MSSubClass", which is stored as a numerical variable but represents discrete building categories and is therefore treated as a categorical feature.

Several features in the dataset contain a high number of missing values. In many cases, these missing values are not random but can be explained by the absence of a corresponding property (e.g., no basement, no garage). Additionally, some sparse features may be partially inferred from related variables.

While some features suffer from missing values, others exhibit high sparsity, meaning that most observations are concentrated at zero or in a single category. These features may lack sufficient variability to contribute meaningful information to the model or lead to misleading importance estimates, as rare values can cause the model to overfit and appear artificially informative and are therefore evaluated critically during feature selection.

In [3]:
# checking for columns with a high percentage of 0 values
zero_percentage = (df == 0).sum() / len(df)
high_zero_columns = zero_percentage[zero_percentage > 0.9].index
print("Columns with more than 90% zero values:")
print(high_zero_columns)

Columns with more than 90% zero values:
Index(['LowQualFinSF', 'BsmtHalfBath', '3SsnPorch', 'ScreenPorch', 'PoolArea',
       'MiscVal'],
      dtype='object')


We will take that into acount when analysing the feature importance, now we will continue the data quality assessment.

In [4]:
# some basic statistics about the dataset
print(df.describe().T[['mean', 'std', 'min', 'max', 'count']])

                        mean           std      min       max   count
Id                730.500000    421.610009      1.0    1460.0  1460.0
MSSubClass         56.897260     42.300571     20.0     190.0  1460.0
LotFrontage        70.049958     24.284752     21.0     313.0  1201.0
LotArea         10516.828082   9981.264932   1300.0  215245.0  1460.0
OverallQual         6.099315      1.382997      1.0      10.0  1460.0
OverallCond         5.575342      1.112799      1.0       9.0  1460.0
YearBuilt        1971.267808     30.202904   1872.0    2010.0  1460.0
YearRemodAdd     1984.865753     20.645407   1950.0    2010.0  1460.0
MasVnrArea        103.685262    181.066207      0.0    1600.0  1452.0
BsmtFinSF1        443.639726    456.098091      0.0    5644.0  1460.0
BsmtFinSF2         46.549315    161.319273      0.0    1474.0  1460.0
BsmtUnfSF         567.240411    441.866955      0.0    2336.0  1460.0
TotalBsmtSF      1057.429452    438.705324      0.0    6110.0  1460.0
1stFlrSF         116

The minimum and maximum values in the "YrSold" column indicate that the data was collected between 2006 and 2010. While this dataset would not be suitable for predicting current house prices, it is appropriate for building a model that predicts house prices in 2010. Therefore, the data is considered adequate with respect to temporal relevance.
The other numerical values are plausible given their minimum and maximum.

In [5]:
# checking for consistent spelling of categorical variables
categorical_columns = df.select_dtypes(include=['object']).columns
for col in categorical_columns:
    unique_values = df[col].unique()
    print(f"Unique values in '{col}': {unique_values}")


Unique values in 'MSZoning': ['RL' 'RM' 'C (all)' 'FV' 'RH']
Unique values in 'Street': ['Pave' 'Grvl']
Unique values in 'Alley': [nan 'Grvl' 'Pave']
Unique values in 'LotShape': ['Reg' 'IR1' 'IR2' 'IR3']
Unique values in 'LandContour': ['Lvl' 'Bnk' 'Low' 'HLS']
Unique values in 'Utilities': ['AllPub' 'NoSeWa']
Unique values in 'LotConfig': ['Inside' 'FR2' 'Corner' 'CulDSac' 'FR3']
Unique values in 'LandSlope': ['Gtl' 'Mod' 'Sev']
Unique values in 'Neighborhood': ['CollgCr' 'Veenker' 'Crawfor' 'NoRidge' 'Mitchel' 'Somerst' 'NWAmes'
 'OldTown' 'BrkSide' 'Sawyer' 'NridgHt' 'NAmes' 'SawyerW' 'IDOTRR'
 'MeadowV' 'Edwards' 'Timber' 'Gilbert' 'StoneBr' 'ClearCr' 'NPkVill'
 'Blmngtn' 'BrDale' 'SWISU' 'Blueste']
Unique values in 'Condition1': ['Norm' 'Feedr' 'PosN' 'Artery' 'RRAe' 'RRNn' 'RRAn' 'PosA' 'RRNe']
Unique values in 'Condition2': ['Norm' 'Artery' 'RRNn' 'Feedr' 'PosN' 'PosA' 'RRAn' 'RRAe']
Unique values in 'BldgType': ['1Fam' '2fmCon' 'Duplex' 'TwnhsE' 'Twnhs']
Unique values in 'Hous

The categorical features appear to be consistent.



Overall, the dataset seems to be well preprocessed, as also indicated in the Kaggle description. The values are consistent, plausible, and provided in appropriate formats.  

Most missing values can be explained by structural reasons rather than random data quality issues, making them suitable for imputation. Additionally, the dataset is temporally appropriate for the intended prediction task.  

Therefore, the overall data quality can be considered good, and we proceed with this dataset for further analysis and for data cleaning and preprocessing steps, this leaves us only the step missing value imputaion.

## 3. Missing Value Analysis and Imputation

In [6]:
# columns with missing values
missing_value_columns = df.columns[df.isnull().any()]
print("Columns with missing values:")
print(missing_value_columns)

Columns with missing values:
Index(['LotFrontage', 'Alley', 'MasVnrType', 'MasVnrArea', 'BsmtQual',
       'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
       'Electrical', 'FireplaceQu', 'GarageType', 'GarageYrBlt',
       'GarageFinish', 'GarageQual', 'GarageCond', 'PoolQC', 'Fence',
       'MiscFeature'],
      dtype='object')


As discussed in the Data Quality Assessment, most features with missing values can be explained or imputed using related features.
The following table summarizes features with missing values and potential explanatory columns:

| Feature | Non-Null Count | Potential Explanation | Related Columns |
|--------|---------------|----------------------|----------------|
| LotFrontage | 1201 | May be inferred from lot characteristics | LotArea, Neighborhood |
| Alley | 91 | Depends on Alley existence | Street, LotShape |
| MasVnrType | 588 | Related to masonry veneer area | MasVnrArea |
| BsmtQual | 1423 | Basement quality depends on basement existence | BsmtFinSF1 |
| BsmtCond | 1423 | Basement condition depends on basement existence | BsmtFinSF1 |
| BsmtExposure | 1422 | Basement exposure depends on basement existence | BsmtFinSF1 |
| BsmtFinType1 | 1423 | Basement finish type depends on basement existence | BsmtFinSF1 |
| BsmtFinType2 | 1422 | Basement finish type depends on basement existence | BsmtFinSF2 |
| Electrical | 1459 | Likely random missing value | — |
| FireplaceQu | 770 | Quality depends on fireplace existence | Fireplaces |
| GarageType | 1379 | Depends on garage existence | GarageCars |
| GarageYrBlt | 1379 | Depends on garage existence | GarageCars |
| GarageFinish | 1379 | Depends on garage existence | GarageCars |
| GarageQual | 1379 | Depends on garage existence | GarageCars |
| GarageCond | 1379 | Depends on garage existence | GarageCars |
| PoolQC | 7 | Depends on presence of a pool | PoolArea |
| Fence | 281 | May indicate absence of a fence | — |
| MiscFeature | 54 | Depends on presence of miscellaneous features | MiscVal |

We now evaluate missing values by comparing columns with missing entries to related columns without missing values in order to identify potential patterns in the missingness.

### 3.1 Structural Missing Value Imputation

In [7]:
# comparing column "LotFrontage", "LotArea", "Neighborhood" to see if there is a pattern in the missingness
df[["LotFrontage", "LotArea", "Neighborhood"]]

,LotFrontage,LotArea,Neighborhood
0,65.0,8450,CollgCr
1,80.0,9600,Veenker
2,68.0,11250,CollgCr
3,60.0,9550,Crawfor
4,84.0,14260,NoRidge
...,...,...,...
1455,62.0,7917,Gilbert
1456,85.0,13175,NWAmes
1457,66.0,9042,Crawfor
1458,68.0,9717,NAmes


LotFrontage represents the linear feet of street connected to the property and is therefore influenced by the layout of the neighborhood rather than the total lot size. To account for this, missing values are imputed using the median within each neighborhood.

In [8]:
# impute missing values in "LotFrontage" with the median value of "LotFrontage" for each "Neighborhood"
df["LotFrontage"] = df.groupby("Neighborhood")["LotFrontage"].transform(lambda x: x.fillna(x.median()))
df["LotFrontage"].isnull().sum()

np.int64(0)

In [9]:
# evaluating missing values in "Alley" by comparing it to "GarageType", "Neighborhood", "LotConfig" and numerical columns 
# "LotArea" and "LotFrontage" to see if there is a pattern in the missingness

# 1. Distribution of Alley
print("Distribution of Alley (including NaN):")
print(df["Alley"].value_counts(dropna=False, normalize=True))

# 2. Relationship with GarageType
print("\nAlley vs GarageType:")
print(pd.crosstab(df["Alley"].notna(), df["GarageType"]))

# 3. Relationship with Neighborhood
print("\nAlley vs Neighborhood:")
print(pd.crosstab(df["Neighborhood"], df["Alley"].notna()))

# 4. Relationship with LotConfig
print("\nAlley vs LotConfig:")
print(pd.crosstab(df["LotConfig"], df["Alley"].notna()))

# 5. Numerical comparison
print("\nNumerical comparison (LotArea, LotFrontage) mean:")
print(df.groupby(df["Alley"].notna())[["LotArea", "LotFrontage"]].mean())

Distribution of Alley (including NaN):
Alley
NaN     0.937671
Grvl    0.034247
Pave    0.028082
Name: proportion, dtype: float64

Alley vs GarageType:
GarageType  2Types  Attchd  Basment  BuiltIn  CarPort  Detchd
Alley                                                        
False            5     855       19       88        9     321
True             1      15        0        0        0      66

Alley vs Neighborhood:
Alley         False  True 
Neighborhood              
Blmngtn          17      0
Blueste           2      0
BrDale           16      0
BrkSide          53      5
ClearCr          28      0
CollgCr         150      0
Crawfor          50      1
Edwards          94      6
Gilbert          79      0
IDOTRR           30      7
MeadowV          17      0
Mitchel          49      0
NAmes           224      1
NPkVill           9      0
NWAmes           73      0
NoRidge          41      0
NridgHt          77      0
OldTown          70     43
SWISU            21      4
Sawyer    

This analysis shows systematic relationships with other features such as "GarageType", "Neighborhood" and "LotConfig", indicating that missing values are not random but reflect the absence of alley access. Therefore, missing values are imputed with a separate category ("NoA").

In [10]:
# imputing missing values in "Alley" with "NoA"
df["Alley"] = df["Alley"].fillna("NoA")
df["Alley"].isnull().sum()

np.int64(0)

In [11]:
# evaluating missing values in "MasVnrType" by comparing it to "MasVnrArea" to see if there is a pattern in the missingness,
# since they are related to each other and the missingness is likely not random

df[["MasVnrType", "MasVnrArea"]].head(10)

,MasVnrType,MasVnrArea
0,BrkFace,196.0
1,NaN,0.0
2,BrkFace,162.0
3,NaN,0.0
4,BrkFace,350.0
5,NaN,0.0
6,Stone,186.0
7,Stone,240.0
8,NaN,0.0
9,NaN,0.0


This shows, that the missing values mean no vaneer, so we impute the missing values with "NoV".

In [12]:
# imputing missing values in "MasVnrType" with "NoV" and in "MasVnrArea" with 0
df["MasVnrType"] = df["MasVnrType"].fillna("NoV")
df["MasVnrArea"] = df["MasVnrArea"].fillna(0)
df["MasVnrType"].isnull().sum()
df["MasVnrArea"].isnull().sum()

np.int64(0)

In [13]:
# evaluating the "Basement" features, by comparing all the "Basement" features with each other to see if there is a pattern in the missingness 
# and if we can impute them with a common value

df[["BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2", "BsmtFinSF1", "BsmtFinSF2"]].loc[
    df["BsmtQual"].isnull() | df["BsmtCond"].isnull() | df["BsmtExposure"].isnull() | df["BsmtFinType1"].isnull() | df["BsmtFinType2"].isnull()
].head(5).T

,17,39,90,102,156
BsmtQual,NaN,NaN,NaN,NaN,NaN
BsmtCond,NaN,NaN,NaN,NaN,NaN
BsmtExposure,NaN,NaN,NaN,NaN,NaN
BsmtFinType1,NaN,NaN,NaN,NaN,NaN
BsmtFinType2,NaN,NaN,NaN,NaN,NaN
BsmtFinSF1,0,0,0,0,0
BsmtFinSF2,0,0,0,0,0


Apparently, all missing values in columns "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1" and "BsmtFinType2" can be imputed with "NoB".

In [14]:
# imputing missing values in columns "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1" and "BsmtFinType2" with "NoB"
df["BsmtQual"] = df["BsmtQual"].fillna("NoB")
df["BsmtCond"] = df["BsmtCond"].fillna("NoB")
df["BsmtExposure"] = df["BsmtExposure"].fillna("NoB")
df["BsmtFinType1"] = df["BsmtFinType1"].fillna("NoB")
df["BsmtFinType2"] = df["BsmtFinType2"].fillna("NoB")

Missing values in "FireplaceQu" might also be related to the existence of a fireplace, so we will compare "FireplaceQu" to "Fireplaces"

In [15]:
# comparing "FireplaceQu" and "Fireplaces" to see if there is a pattern in the missingness and if we can impute them with a common value
df[["FireplaceQu", "Fireplaces"]].loc[df["FireplaceQu"].isnull()].head(10)

,FireplaceQu,Fireplaces
0,NaN,0
5,NaN,0
10,NaN,0
12,NaN,0
15,NaN,0
17,NaN,0
18,NaN,0
19,NaN,0
26,NaN,0
29,NaN,0


So we can impute missing values in "FireplaceQu" with category "NoF".

In [16]:
# imputing missing values in "FireplaceQu" with category "NoF"
df["FireplaceQu"] = df["FireplaceQu"].fillna("NoF")

We can already assume that all missing "Garage" values are based on the absence of a garage, we will now confirm that.

In [17]:
# evaluating the "Garage" features, by comparing all the "Garage" features with each other to see if there is a pattern in the missingness 
# and if we can impute them with a common value

Garage_features = ["GarageType", "GarageYrBlt", "GarageFinish", "GarageCars", "GarageCond", "GarageQual"]

df[Garage_features].loc[
    df["GarageType"].isnull() | df["GarageYrBlt"].isnull() | df["GarageFinish"].isnull() | df["GarageCars"].isnull() 
    | df["GarageCond"].isnull() | df["GarageQual"].isnull()
].head(5).T

,39,48,78,88,89
GarageType,NaN,NaN,NaN,NaN,NaN
GarageYrBlt,NaN,NaN,NaN,NaN,NaN
GarageFinish,NaN,NaN,NaN,NaN,NaN
GarageCars,0,0,0,0,0
GarageCond,NaN,NaN,NaN,NaN,NaN
GarageQual,NaN,NaN,NaN,NaN,NaN


The assumption was correct, we can imputate all categorical missing "Garage" values with "NoG". For the numerical feature "GarageYrBlt", missing values also correspond to houses without a garage. To preserve this information, an additional binary feature ("HasGarage") is introduced. The missing values in "GarageYrBlt" are then imputed with 0.

In [18]:
# impute all missing "Garage" values with "NoG".
df["GarageType"] = df["GarageType"].fillna("NoG")
df["GarageFinish"] = df["GarageFinish"].fillna("NoG")
df["GarageCond"] = df["GarageCond"].fillna("NoG")
df["GarageQual"] = df["GarageQual"].fillna("NoG")

# additional feature "HasGarage" to indicate whether the house has a garage or not, 
# since "GarageYrBlt" has some missing values that we impute with 0, which could be misleading, 
# because it could be interpreted as a garage built in the year 0, which is not possible

df["HasGarage"] = (df["GarageCars"] > 0).astype(int)
df["GarageYrBlt"] = df["GarageYrBlt"].fillna(0)

The feature "Fence" contains a large proportion of missing values. Given the nature of the feature and the distribution of the data, it is reasonable to assume that missing values indicate the absence of a fence rather than missing data. While this assumption cannot be verified directly, it is consistent with similar features in the dataset where missing values represent the absence of a property. So we will fill missing values in "Fence" with category "NoF"

In [19]:
# impute missing values in "Fence" with "NoF"
df["Fence"] = df["Fence"].fillna("NoF")

### 3.2 Statistical Missing Value Imputation

The feature "Electrical" contains only a single missing value. Since this represents a negligible proportion of the dataset, it is imputed using the most frequent category (mode).

In [20]:
# imputating missing value in "Electrical" with the most common value "SBrkr", since there is only one missing value and it is likely not random
df["Electrical"] = df["Electrical"].fillna("SBrkr")

### 3.3 Dropped Features

The feature "PoolQC" provides information about pool quality but contains only a very small number of observations. Due to the extremely low sample size, this feature is likely to introduce noise and overfitting. Instead, the numerical feature "PoolArea" is retained, as it captures the presence and size of a pool more robustly and "PoolQC" is removed from the dataset.

In [21]:
# dropping "PoolQC"
df = df.drop(columns=["PoolQC"])

The feature "MiscFeature" contains a very high proportion of missing values and only a few rare, heterogeneous categories. Due to the lack of a consistent structure and the extremely low frequency of observed values, this feature is unlikely to provide meaningful predictive information and may lead to overfitting. Therefore, it is removed from the dataset. As a consequence, WMiscVal" loses its meaning without the corresponding feature and is also removed.

In [22]:
# drop "MiscFeature" and "MiscVal"
df = df.drop(columns=["MiscFeature", "MiscVal"])

Altough there is no missing value, the "Id" coulmn ist not relevant for our work and therefore will be dropped, too.

In [23]:
# dropping "Id"
df = df.drop(columns=["Id"])

### 3.4 Final Missing Value Check

Now we should have handled all missing values. 

In [24]:
# columns with missing values
missing_value_columns = df.columns[df.isnull().any()]
print("Columns with missing values:")
print(missing_value_columns)

Columns with missing values:
Index([], dtype='object')


## 4. Feature Engineering
After handling missing values, the dataset is prepared for further transformations. In this step, categorical features are encoded appropriately depending on their nature.

In addition, the analysis of individual features revealed meaningful relationships between variables. Based on these insights, several new features are constructed to better capture underlying patterns in the data and to provide more informative representations for the models.

### 4.1 Encoding categorical Features

We now encode the features to enable a unified importance measure across all variables, which will later be used to identify the most influential features for predicting "SalePrice". Categorical features are encoded according to their nature:  
Nominal features without an inherent order are transformed using one-hot encoding, while ordinal features with a natural ranking are encoded using mapping to preserve their order.  
The resulting encoded dataset is also required for applying machine learning algorithms in the subsequent modeling stage.

In [25]:
# analyzing the average SalePrice across its categories to decide which features have nominal values and which features have ordinal values, 
# since this will affect the choice of encoding method for these features in the next steps of the project
categorical_features = df.select_dtypes(include=['object']).columns
target = "SalePrice"

for feature in categorical_features:
    print(f"\nAverage SalePrice by {feature}:")
    print(df.groupby(feature)[target].mean())


Average SalePrice by MSZoning:
MSZoning
C (all)     74528.000000
FV         214014.061538
RH         131558.375000
RL         191004.994787
RM         126316.830275
Name: SalePrice, dtype: float64

Average SalePrice by Street:
Street
Grvl    130190.500000
Pave    181130.538514
Name: SalePrice, dtype: float64

Average SalePrice by Alley:
Alley
Grvl    122219.080000
NoA     183452.131483
Pave    168000.585366
Name: SalePrice, dtype: float64

Average SalePrice by LotShape:
LotShape
IR1    206101.665289
IR2    239833.365854
IR3    216036.500000
Reg    164754.818378
Name: SalePrice, dtype: float64

Average SalePrice by LandContour:
LandContour
Bnk    143104.079365
HLS    231533.940000
Low    203661.111111
Lvl    180183.746758
Name: SalePrice, dtype: float64

Average SalePrice by Utilities:
Utilities
AllPub    180950.95682
NoSeWa    137500.00000
Name: SalePrice, dtype: float64

Average SalePrice by LotConfig:
LotConfig
Corner     181623.425856
CulDSac    223854.617021
FR2        177934.5744

It appears that variables ending with "Qual" or similar (QC, Qu, Cond) represent ordinal features, as they describe quality levels with a natural ordering (e.g., Poor < Fair < Typical < Good < Excellent), just like the features "BsmtExposure", "BsmtFinType1", "BsmtFinType2", "Fence", "Functional". Similarly, "GarageFinish" can be considered ordinal, as it reflects increasing levels of completion.  

In contrast, "Neighborhood" is treated as a nominal variable, since it does not have a natural or consistent ordering, even though some neighborhoods may generally be considered more desirable than others. The same goes for "MSZoning", "Neighborhood", "Condition1", "Condition2", "BldgType", "HouseStyle", "RoofStyle", "RoofMatl", "Exterior1st", "Exterior2nd", "Foundation", "Heating", "CentralAir", "Electrical", "GarageType", "PavedDrive", "LotConfig", "LandContour", "Utilities", "LandSlope", "Street", "Alley", "SaleType", "SaleCondition", "LotShape" and "MasVnrType".
While an ordering can also be observed for e.g. "Neighborhood", it is data-driven rather than inherent, and the feature is therefore treated as nominal.

One special case is "MSSubClass". Although it is represented by numerical values, these values correspond to categories of building classes rather than continuous quantities. Moreover, these categories do not have an inherent order, which is why "MSSubClass" is treated as a nominal feature.

This distinction is important, as ordinal features can be encoded using ordered mappings, whereas nominal features require one-hot encoding, which we will do now.

In [26]:
# creating an encoded DataFrame
df_encoded = df.copy()

# 1. Ordinal encoding
# Ordinal features have a natural ranking and are therefore mapped to ordered numerical values.

ordinal_mapping = {
    "ExterQual": {"Fa": 0, "TA": 1, "Gd": 2, "Ex": 3},
    "ExterCond": {"Po":0, "Fa": 1, "TA": 2, "Gd": 3, "Ex": 4},
    "BsmtQual": {"NoB": 0, "Fa": 1, "TA": 2, "Gd": 3, "Ex": 4},
    "BsmtCond": {"Po":0, "NoB": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "HeatingQC": {"Po": 0, "Fa": 1, "TA": 2, "Gd": 3, "Ex": 4},
    "KitchenQual": {"Fa": 0, "TA": 1, "Gd": 2, "Ex": 3},
    "FireplaceQu": {"Po": 0, "NoF": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "GarageQual": {"Po":0,"NoG": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "GarageCond": {"Po": 0,"NoG": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "GarageFinish": {"NoG": 0, "Unf": 1, "RFn": 2, "Fin": 3},
    "BsmtExposure": {"NoB": 0, "No": 1, "Mn": 2, "Av": 3, "Gd": 4},
    "BsmtFinType1": {"NoB": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6},
    "BsmtFinType2": {"NoB": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6},
    "Fence": {"NoF": 0, "MnWw": 1, "GdWo": 2, "MnPrv": 3, "GdPrv": 4},
    "Functional": {"Sal": 0, "Sev": 1, "Maj2": 2, "Maj1": 3, "Mod": 4, "Min2": 5, "Min1": 6, "Typ": 7}
}


ordinal_features = [col for col in ordinal_mapping if col in df_encoded.columns]

for col in ordinal_features:
    df_encoded[col] = df_encoded[col].map(ordinal_mapping[col])



# 2. One-hot encoding for nominal features
# Nominal features do not have a natural order and are therefore encoded as binary dummy variables.

nominal_features = [
    "MSZoning", "Neighborhood", "Condition1", "Condition2",
    "BldgType", "HouseStyle", "RoofStyle", "RoofMatl",
    "Exterior1st", "Exterior2nd", "Foundation",
    "Heating", "CentralAir", "Electrical",
    "GarageType", "PavedDrive",
    "LotConfig", "LandContour", "Utilities", "LandSlope",
    "Street", "Alley", "SaleType", "SaleCondition",
    "LotShape", "MasVnrType"
]

# 2.1 Handling special case MSSubClass 
if "MSSubClass" in df_encoded.columns:
    nominal_features.append("MSSubClass")

nominal_features = [col for col in nominal_features if col in df_encoded.columns]

df_encoded = pd.get_dummies(df_encoded, columns=nominal_features, drop_first=False)

df_encoded.shape


(1460, 242)

### 4.2 Constructing Features
We will now use the primaly learned insight about relationships between variables to construct new features:
- house age
- recently remodeled or build: weather it was remodeled or build in the last 15 years
- total living ares (Basement and above ground)
- total bathrooms (in basement, half and full bathrooms)

In [27]:
# adding feature "house_age"
df_encoded["house_age"] = df_encoded["YrSold"] - df_encoded["YearBuilt"]


# adding feature "recently_remodeled_or_built" if it was remodeles or built in the last 15 years
df_encoded["recently_remodeled_or_built"] = (
    ((df_encoded["YearRemodAdd"] >= df_encoded["YrSold"] - 15) & 
     (df_encoded["YearRemodAdd"] <= df_encoded["YrSold"])) |
    
    ((df_encoded["YearBuilt"] >= df_encoded["YrSold"] - 15) & 
     (df_encoded["YearBuilt"] <= df_encoded["YrSold"]))
).astype(int)

# adding feature "total_living_area"
df_encoded["total_living_area"] = df_encoded["TotalBsmtSF"] + df_encoded["GrLivArea"]


# adding feature "total_bathrooms" by combining "FullBath", "HalfBath", "BsmtFullBath" and "BsmtHalfBath"
df_encoded["total_bathrooms"] = df_encoded["BsmtFullBath"] + 0.5*df_encoded["BsmtHalfBath"] + df_encoded["FullBath"] + 0.5*df_encoded["HalfBath"]


## 5. Feature Selection

We now have all features in a numerical format, allowing us to evaluate the importance of each feature on a unified scale. To capture both linear and non-linear relationships between the features and the target variable, we use Mutual Information.

Since one-hot encoding splits categorical features into multiple binary columns, we aggregate these columns to recover the importance of the original feature. This is done by summing the Mutual Information values of all subcolumns belonging to the same original feature.

In [28]:
# Compute mutual information on encoded feature level
from sklearn.feature_selection import mutual_info_regression

X = df_encoded.drop(columns=[target])
y = df_encoded[target]

mi_scores = mutual_info_regression(X, y, random_state=42)

mi_sorted = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)



# Aggregate one-hot encoded columns back to original feature level

def get_original_feature_name(encoded_col, original_nominal_features):
    """
    Map encoded column names back to their original feature names.
    For one-hot encoded columns, the original feature name is the
    matching nominal feature prefix. Otherwise, the column name is returned unchanged.
    """
    for feature in original_nominal_features:
        prefix = f"{feature}_"
        if encoded_col.startswith(prefix):
            return feature
    return encoded_col


grouped_scores = {}

for col, score in mi_sorted.items():
    original_feature = get_original_feature_name(col, nominal_features)
    grouped_scores.setdefault(original_feature, []).append(score)

# Aggregation strategy:
# sum(...) is used so that the total information contribution of all categories belonging to one original feature is preserved.

mi_grouped = pd.Series({feature: sum(scores) for feature, scores in grouped_scores.items()}).sort_values(ascending=False)

print("\nTop 20 features by aggregated mutual information:")
print(mi_grouped.head(20))

# Use the aggregated ranking to select the most relevant features.

top_features = mi_grouped.head(20).index.tolist()
all_features = mi_grouped.index.tolist()

print("\nSelected features:")
print(top_features)

print("\nLeast relevant features by aggregated mutual information:")
print(mi_grouped.tail(5))

# Build final encoded modeling dataset from selected features. Keep all encoded columns that belong to the selected features.

selected_encoded_columns = []

for col in X.columns:
    original_feature = get_original_feature_name(col, nominal_features)
    if original_feature in top_features:
        selected_encoded_columns.append(col)

df_enc_sel = df_encoded[selected_encoded_columns + [target]].copy()

print("\nFinal modeling dataset shape:")
print(df_enc_sel.shape)


Top 20 features by aggregated mutual information:
total_living_area    0.664069
Neighborhood         0.605364
OverallQual          0.561972
GrLivArea            0.480435
MSSubClass           0.371711
TotalBsmtSF          0.367657
GarageCars           0.364991
YearBuilt            0.360560
GarageArea           0.359095
total_bathrooms      0.351131
house_age            0.339537
ExterQual            0.324698
KitchenQual          0.321469
Foundation           0.320530
BsmtQual             0.316643
GarageType           0.310082
1stFlrSF             0.309409
FullBath             0.261190
YearRemodAdd         0.257866
GarageFinish         0.255798
dtype: float64

Selected features:
['total_living_area', 'Neighborhood', 'OverallQual', 'GrLivArea', 'MSSubClass', 'TotalBsmtSF', 'GarageCars', 'YearBuilt', 'GarageArea', 'total_bathrooms', 'house_age', 'ExterQual', 'KitchenQual', 'Foundation', 'BsmtQual', 'GarageType', '1stFlrSF', 'FullBath', 'YearRemodAdd', 'GarageFinish']

Least relevant featur

The feature importance analysis indicates that "total_living_area", "OverallQual" and "Neighborhood" are the dominant predictors in the model. This aligns well with intuition, as living space and overall quality are fundamental characteristics that directly influence a property's value.

The importance of "Neighborhood" is also expected, as property prices are strongly driven by location. Neighborhoods differ in terms of proximity to city centers, infrastructure, and socio-economic conditions. Moreover, houses within the same neighborhood often exhibit similarities in architecture and quality due to local building regulations and shared development patterns.

This results in a reinforcing relationship: larger and higher-quality houses are more likely to be located in more expensive neighborhoods, while these neighborhoods themselves tend to maintain a certain standard of construction and quality. Consequently, "Total_living_area*, "OverallQual", and "Neighborhood" emerge as the most influential features in predicting house prices.

Other features such as "GrLivArea" and "TotalBsmtSF" also contribute significantly, highlighting the importance of living space and basement size. Larger properties generally provide more usable space, which increases their market value. These features, "GrLivArea" and "TotalBsmtSF", are likely correlated as they both capture aspects of the overall property size. In tree-based models however, this does not pose a major issue, as the model can select the most informative feature for splitting. 

In contrast, the features "3SsnPorch", "BsmtFinSF2" and "MoSold" provide little additional information for predicting the sale price.
"BsmtFinSF2" is already implicitly captured in the more informative feature "TotalBsmtSF" and appears to have limited predictive power on its own.
The feature "3SsnPorch" is highly sparse, confirming the earlier observation that sparse features either lead to unstable importance estimates or contain very little useful information.  
In this case, the near-zero importance indicates that the feature does not contribute meaningfully to the prediction.
Finally, "MoSold" does not describe any intrinsic property of the house and does not capture broader market trends, which are better reflected by the year of sale ("YrSold").  
Therefore, it is intuitive that the month of sale has little impact on the sale price.

## 6. Train-Test Split

The dataset is split into training and test sets to evaluate model performance on unseen data. We use a standard 80/20 split, where 80% of the data is used for training and 20% for testing. A fixed random state is set to ensure reproducibility of the results.

In [29]:
# splitting the data into training and testing sets
from sklearn.model_selection import train_test_split
X = df_enc_sel.drop(columns=[target])
y = df_enc_sel[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## 7. Model Training and Baseline Evaluation

We compare four different models — Decision Tree Regressor, Random Forest Regressor, Gradient Boost Regressor and Linear Regression — to evaluate whether the relationship between the features and the target variable is better captured by linear or non-linear models.  

Model performance is evaluated using RMSE and R² on the test set. To assess potential overfitting, we also compare the RMSE on the training and test data. A large gap between these values indicates that the model may not generalize well to unseen data.  

The Decision Tree serves as an interpretable non-linear model, the Random Forest as a more robust ensemble method, the Gradient Boosting Regressor as a potentially more powerful model and Linear Regression as a baseline for linear relationships.

In [30]:
import numpy as np
# training decision tree regressor
from sklearn.tree import DecisionTreeRegressor
dtr = DecisionTreeRegressor(random_state=42)
dtr.fit(X_train, y_train)

# training random forest regressor
from sklearn.ensemble import RandomForestRegressor
rfr = RandomForestRegressor(random_state=42)
rfr.fit(X_train, y_train)

# training gradient boosting regressor
from sklearn.ensemble import GradientBoostingRegressor
gbr = GradientBoostingRegressor(random_state=42)
gbr.fit(X_train, y_train)

# training linear regression model
from sklearn.linear_model import LinearRegression
lr = LinearRegression()
lr.fit(X_train, y_train)

# evaluating each model on the test set
from sklearn.metrics import root_mean_squared_error, r2_score

# Decision Tree Regressor
y_pred_dtr = dtr.predict(X_test) 
y_pred_dtr_train = dtr.predict(X_train)
rmse_dtr = root_mean_squared_error(y_test, y_pred_dtr)
rmse_dtr_train = root_mean_squared_error(y_train, y_pred_dtr_train)
r2_dtr = r2_score(y_test, y_pred_dtr)
rmse_gap_dtr = rmse_dtr - rmse_dtr_train

# Random Forest Regressor
y_pred_rfr = rfr.predict(X_test)
y_pred_rfr_train = rfr.predict(X_train)
rmse_rfr = root_mean_squared_error(y_test, y_pred_rfr)
rmse_rfr_train = root_mean_squared_error(y_train, y_pred_rfr_train)
r2_rfr = r2_score(y_test, y_pred_rfr)
rmse_gap_rfr = rmse_rfr - rmse_rfr_train

# Gradient Boosting Regressor
y_pred_gbr = gbr.predict(X_test)
y_pred_gbr_train = gbr.predict(X_train)
rmse_gbr = root_mean_squared_error(y_test, y_pred_gbr)
rmse_gbr_train = root_mean_squared_error(y_train, y_pred_gbr_train)
r2_gbr = r2_score(y_test, y_pred_gbr)
rmse_gap_gbr = rmse_gbr - rmse_gbr_train

# Linear Regression
y_pred_lr = lr.predict(X_test)
y_pred_lr_train = lr.predict(X_train)
rmse_lr = root_mean_squared_error(y_test, y_pred_lr)
rmse_lr_train = root_mean_squared_error(y_train, y_pred_lr_train)
r2_lr = r2_score(y_test, y_pred_lr)
rmse_gap_lr = rmse_lr - rmse_lr_train

# printing a table comparing the performance of all models to analyze which model performs best on the final selected features
performance_comparison = pd.DataFrame({
    'Model': ['Decision Tree Regressor', 'Random Forest Regressor', 'Gradient Boosting Regressor', 'Linear Regression'],
    'RMSE': [rmse_dtr, rmse_rfr, rmse_gbr, rmse_lr],
    'R^2 Score': [r2_dtr, r2_rfr, r2_gbr, r2_lr],
    'RMSE Gap': [rmse_gap_dtr, rmse_gap_rfr, rmse_gap_gbr, rmse_gap_lr]
})

print("\nPerformance Comparison of Models:")
print(performance_comparison)


Performance Comparison of Models:
                         Model          RMSE  R^2 Score      RMSE Gap
0      Decision Tree Regressor  36519.226480   0.826128  35980.806079
1      Random Forest Regressor  29692.584240   0.885057  18465.554387
2  Gradient Boosting Regressor  28760.857087   0.892158  12144.735741
3            Linear Regression  34137.360478   0.848069   3681.557614


The performance comparison shows that the Gradient Boosting Regressor is the most powerful non-linear model. With an RMSE of 28,761 and an R² score of 0.892, it outperforms the other models and, with an RMSE gap of 12,145, also demonstrates better generalization (compared to the non-linear models). This highlights the advantage of sequential ensemble methods in capturing complex patterns within the data. 

The Random Forest Regressor performs very similarly to the Gradient Boosting Regressor but is consistently slightly inferior across all evaluation metrics.

In contrast, the Decision Tree Regressor performs worst among the non-linear models, indicating its limited predictive capability in this setting.

After the initial model comparison, we focus on optimizing the most promising models by tuning the hyperparameters of the Gradient Boosting Regressor and Random Forest Regressor to further improve their predictive performance.

## 8. Model Improvement

In the baseline evaluation we decided to tune the hyperparameters of Gradient Boosting Regressor and Random Forest, to compare optimal models of these two and Linear Regression. That's what we will do now.

In [31]:
# tuning the best performing models (Random Forest Regressor and Gradient Boosting Regressor) using GridSearchCV to find 
# the optimal hyperparameters and improve the model's performance
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score

# Define the parameter grid for Gradient Boosting Regressor
param_grid_gbr = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 10],
    'learning_rate': [0.1, 0.05, 0.01],
    'subsample': [0.8, 1.0]
}

# Define the parameter grid for Random Forest Regressor
param_grid_rfr = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

# GridSearchCV for Gradient Boosting Regressor
grid_search_gbr = GridSearchCV(estimator=gbr, param_grid=param_grid_gbr, cv=5, n_jobs=-1)
grid_search_gbr.fit(X_train, y_train)

# GridSearchCV for Random Forest Regressor
grid_search_rfr = GridSearchCV(estimator=rfr, param_grid=param_grid_rfr, cv=5, n_jobs=-1)
grid_search_rfr.fit(X_train, y_train)

# Best hyperparameters and performance of the tuned Gradient Boosting Regressor
best_gbr = grid_search_gbr.best_estimator_
y_pred_gbr_tuned = best_gbr.predict(X_test)
y_pred_gbr_tuned_train = best_gbr.predict(X_train)
rmse_gbr_tuned = np.sqrt(mean_squared_error(y_test, y_pred_gbr_tuned))
r2_gbr_tuned = r2_score(y_test, y_pred_gbr_tuned)
rmse_gap_gbr_tuned = rmse_gbr_tuned - np.sqrt(mean_squared_error(y_train, y_pred_gbr_tuned_train))

# Best hyperparameters and performance of the tuned Random Forest Regressor
best_rfr = grid_search_rfr.best_estimator_
y_pred_rfr_tuned = best_rfr.predict(X_test)
y_pred_rfr_tuned_train = best_rfr.predict(X_train)
rmse_rfr_tuned = np.sqrt(mean_squared_error(y_test, y_pred_rfr_tuned))
r2_rfr_tuned = r2_score(y_test, y_pred_rfr_tuned)
rmse_gap_rfr_tuned = rmse_rfr_tuned - np.sqrt(mean_squared_error(y_train, y_pred_rfr_tuned_train))

# printing the performance of the tuned models and linear regression for comparison in a table
tuned_performance_comparison = pd.DataFrame({
    'Model': ['Gradient Boosting Regressor (Tuned)', 'Random Forest Regressor (Tuned)', 'Linear Regression'],
    'RMSE': [rmse_gbr_tuned, rmse_rfr_tuned, rmse_lr],
    'R^2 Score': [r2_gbr_tuned, r2_rfr_tuned, r2_lr],
    'RMSE Gap': [rmse_gap_gbr_tuned, rmse_gap_rfr_tuned, rmse_gap_lr]
})
print("\nPerformance Comparison of Tuned Models:")
print(tuned_performance_comparison)




Performance Comparison of Tuned Models:
                                 Model          RMSE  R^2 Score      RMSE Gap
0  Gradient Boosting Regressor (Tuned)  28259.265067   0.895886  14875.388613
1      Random Forest Regressor (Tuned)  29518.764613   0.886399  16545.836712
2                    Linear Regression  34137.360478   0.848069   3681.557614


Hyperparameter tuning leads to a more accurate Gradient Boosting Regressor, while slightly reducing its generalization ability, reflecting the typical bias-variance trade-off. 
The Random Forest Regressor shows little improvement in accuracy and leads to a better generalization (RMSE gap) after tuning. This suggests both models were close to its optimal performance under default settings, but had different metrics, which could still be improved by hyperparameter training.

## 9. Model Selection

The evaluation of the tuned models shows that the Gradient Boosting Regressor outperforms the Random Forest Regressor across all metrics.
Compared to Linear Regression, it achieves significantly higher predictive accuracy. However, this improvement comes at the cost of reduced generalization performance, as indicated by a larger gap between training and test error.

Based on these results, the Gradient Boosting Regressor appears to be the most promising model. Altough due to the observed trade-off between accuracy and generalization, further validation is required before making a final decision. Therefore, cross-validation is applied in the next step to assess the stability and robustness of all models across different data splits.

In [32]:
# cross validation to evaluate the stability of the tuned models and compare it to linear regression
from sklearn.model_selection import cross_val_score

# Cross-validation for Gradient Boosting Regressor (Tuned) and convert the negative RMSE scores to 
# positive RMSE scores for better interpretability
cv_scores_gbr_tuned = cross_val_score(best_gbr, X, y, cv=5, scoring="neg_root_mean_squared_error")
cv_scores_gbr_tuned = -cv_scores_gbr_tuned  

# Cross-validation for Random Forest Regressor (Tuned) and convert the negative RMSE scores to 
# positive RMSE scores for better interpretability
cv_scores_rfr_tuned = cross_val_score(best_rfr, X, y, cv=5, scoring="neg_root_mean_squared_error")
cv_scores_rfr_tuned = -cv_scores_rfr_tuned  

# Cross-validation for Linear Regression and convert the negative RMSE scores to positive 
# RMSE scores for better interpretability
cv_scores_lr = cross_val_score(lr, X, y, cv=5, scoring="neg_root_mean_squared_error")
cv_scores_lr = -cv_scores_lr  

# printing the cross-validation results in a table for comparison
cv_comparison = pd.DataFrame({
    "Gradient Boosting (Tuned)": cv_scores_gbr_tuned,
    "Random Forest (Tuned)": cv_scores_rfr_tuned,
    "Linear Regression": cv_scores_lr
})

print("\nCross-Validation RMSE Scores:")
print(cv_comparison)
print("\nCross-Validation statistics:")
print(cv_comparison.describe())


Cross-Validation RMSE Scores:
   Gradient Boosting (Tuned)  Random Forest (Tuned)  Linear Regression
0               23298.567461           25895.803190       26016.687229
1               34836.952784           38617.657409       33580.159157
2               25600.502745           29246.569535       34878.502301
3               21920.563200           22905.685631       29673.010874
4               25851.529546           32744.132608       41834.746370

Cross-Validation statistics:
       Gradient Boosting (Tuned)  Random Forest (Tuned)  Linear Regression
count                   5.000000               5.000000           5.000000
mean                26301.623147           29881.969675       33196.621186
std                  5043.759406            6112.814664        5948.956072
min                 21920.563200           22905.685631       26016.687229
25%                 23298.567461           25895.803190       29673.010874
50%                 25600.502745           29246.569535       3

The cross-validation results confirm the strong predictive performance of the Gradient Boosting Regressor, achieving the lowest average RMSE (26,302) compared to the Random Forest Regressor (29,882) and Linear Regression (33,197).

In terms of stability, the standard deviation of the Gradient Boosting Regressor (5,044) is lower than that of the Random Forest Regressor (6,113) and  Linear Regression (5,949). This suggests that, contrary to our previous observations, the Gradient Boosting Regressor actually demonstrates better generalization performance than the other models. As a result, it now outperforms them across all evaluation metrics.

Overall, the cross-validation results confirm the earlier trend and provide a solid basis for selecting the Gradient Boosting Regressor as the final model. Its consistently strong performance across all folds demonstrates both high predictive accuracy and good generalization, thereby justifying this choice.

## 10. Excursus: Explainability – Extracting Decision Rules

While more complex models achieved better predictive performance, decision trees offer a key advantage: interpretability. Therefore, we use a simplified tree model to extract intuitive decision rules that allow for a rough, manual estimation of house prices.

A standard regression tree predicts exact numerical values, which are often too specific to derive general, human-readable insights. To address this, the continuous target variable (SalePrice) is transformed into discrete price categories (e.g., Low, Medium, High).

Instead of using a regression tree, we train a DecisionTreeClassifier on these categories. This allows the model to directly learn class-based decision boundaries, resulting in clearer and more interpretable rules.

By grouping prices into broader intervals and using a shallow decision tree, we obtain simple, transparent rules that capture the main structure of the data. While this approach sacrifices predictive accuracy, it enables intuitive, ad-hoc reasoning without requiring a full machine learning pipeline.

In [33]:
from sklearn.tree import DecisionTreeClassifier, export_text


# setting price category thresholds
price_thresholds = [0, 100000, 200000, 300000, np.inf]
price_labels = ['Very Low', 'Low', 'Medium', 'High']

# create categorical target
y_train_cat = pd.cut(y_train, bins=price_thresholds, labels=price_labels)

# train simple decision tree classifier
dtc_simple = DecisionTreeClassifier(max_depth=3, random_state=42)
dtc_simple.fit(X_train, y_train_cat)

# extract decision rules
decision_rules = export_text(dtc_simple, feature_names=list(X_train.columns))

print("\nDecision rules from the simple decision tree classifier:")
print(decision_rules)



Decision rules from the simple decision tree classifier:
|--- total_living_area <= 2929.50
|   |--- total_living_area <= 1618.00
|   |   |--- YearRemodAdd <= 1976.00
|   |   |   |--- class: Very Low
|   |   |--- YearRemodAdd >  1976.00
|   |   |   |--- class: Low
|   |--- total_living_area >  1618.00
|   |   |--- total_living_area <= 2537.00
|   |   |   |--- class: Low
|   |   |--- total_living_area >  2537.00
|   |   |   |--- class: Low
|--- total_living_area >  2929.50
|   |--- OverallQual <= 6.50
|   |   |--- OverallQual <= 5.50
|   |   |   |--- class: Low
|   |   |--- OverallQual >  5.50
|   |   |   |--- class: Low
|   |--- OverallQual >  6.50
|   |   |--- total_living_area <= 3460.00
|   |   |   |--- class: Medium
|   |   |--- total_living_area >  3460.00
|   |   |   |--- class: High



This simplified decision tree yields the following decision rules:

* total_living_area ≤ 1618 AND YearRemodAdd ≤ 1976 ⇒ Price: Very Low (< $100,000)
* total_living_area ≤ 1618 AND YearRemodAdd > 1976 ⇒ Price: Low ($100,000–200,000)
* total_living_area > 1618 AND total_living_area ≤ 2929.5 ⇒ Price: Low ($100,000–200,000)
* total_living_area > 2929.5 AND OverallQual ≤ 6.5 ⇒ Price: Low ($100,000–200,000)
* total_living_area > 2929.5 AND total_living_area ≤ 3460 AND OverallQual > 6.5 ⇒ Price: Medium ($200,000–300,000)
* total_living_area > 3460 AND OverallQual > 6.5 ⇒ Price: High (> $300,000)

These rules highlight several key insights:

* Total living area is the primary driver of house prices.

  * Medium-sized houses (total_living_area between 1618 and 2929.5) are predominantly priced in the low range ($100,000–200,000).

* For smaller houses (total_living_area ≤ 1618), the year of remodeling becomes decisive:

  * Houses remodeled in 1976 or earlier tend to fall into the very low price category (< $100,000).
  * Houses remodeled after 1976 are typically priced similarly to medium-sized houses ($100,000–200,000).

* For larger houses (total_living_area > 2929.5), overall quality becomes the key differentiator:

  * Lower overall quality (≤ 6.5) limits prices to the low range.
  * Higher overall quality (> 6.5) leads to medium price levels ($200,000–300,000).

* Very large (> 3460) and high-quality (> 6.5) houses reach the highest price category (> $300,000).

This leads us to four basic rules:
* total_living_area ≤ 1618 AND YearRemodAdd ≤ 1976 ⇒ Price: Very Low (< $100,000)
* (total_living_area ≤ 1618 AND YearRemodAdd > 1976) OR (total_living_area > 1618 AND total_living_area ≤ 2929.5) OR (total_living_area > 2929.5 AND OverallQual ≤ 6.5) ⇒ Price: Low ($100,000–200,000)
* total_living_area > 2929.5 AND total_living_area ≤ 3460 AND OverallQual > 6.5 ⇒ Price: Medium ($200,000–300,000)
* total_living_area > 3460 AND OverallQual > 6.5 ⇒ Price: High (> $300,000)


## 11. Pipeline-Based Re-Implementation and Comparison

The previous sections followed a manual, step-by-step workflow that allowed for flexible, data-driven decisions but lacked full reproducibility.
To address this, the modeling process is re-implemented using scikit-learn pipelines, ensuring that all preprocessing steps are applied consistently.

First, a baseline pipeline is constructed using general best practices without incorporating dataset-specific insights. 
In a second step, an optimized pipeline is developed that integrates the knowledge gained from the manual analysis.

This setup enables a direct comparison between manual, purely technical, and combined approaches.




### 11.1 General Best-Practice Pipeline

This first approach represents a purely technical, “blind” workflow that follows general machine learning best practices without incorporating dataset-specific insights.

The pipeline includes standard preprocessing steps: numerical features are imputed using a statistical strategy, while categorical features are imputed and encoded using one-hot encoding. These steps are applied uniformly, without considering the semantic meaning of individual features.

The preprocessing is followed by model training using a Gradient Boosting Regressor. In addition, hyperparameters are tuned using a general grid search with standard parameter ranges, ensuring a fair and robust baseline without relying on data-specific assumptions.


In [34]:
# building "blind" general pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

# loading original dataframe
df_orig = pd.read_csv("train_housing_prices.csv") 

# preparing dataset
X = df_orig.drop(columns="SalePrice")
y = df_orig["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# identify categorical and numerical features
cat_features = [col for col in X.select_dtypes("object").columns]
num_features = [col for col in X.columns if col not in cat_features]

# preprocessing
preprocess = ColumnTransformer([
    ("imputer_num", SimpleImputer(strategy="median"), num_features),
    ("imp_enc_cat", Pipeline([
         ("impute_cat", SimpleImputer(strategy="most_frequent")),
        ("enc_ord", OneHotEncoder(handle_unknown="ignore")),
    ]), cat_features)
])

# building pipeline
gen_pipe = Pipeline([
    ("preprocess", preprocess),
    ("Gradient_Boosting_Regressor", GradientBoostingRegressor(random_state=42))
])

# tuning hyperparameters
param_grid = {
    "Gradient_Boosting_Regressor__n_estimators": [100, 200],
    "Gradient_Boosting_Regressor__max_depth": [3, 5, 10],
    "Gradient_Boosting_Regressor__learning_rate": [0.1, 0.05, 0.01],
    "Gradient_Boosting_Regressor__subsample": [0.8, 1.0]
}

grid_search_pipe = GridSearchCV(estimator=gen_pipe, param_grid=param_grid, cv=5, n_jobs=-1)
grid_search_pipe.fit(X_train, y_train)

# best hyperparameter and performance
gen_pipe = grid_search_pipe.best_estimator_
y_pred_gen_pipe = gen_pipe.predict(X_test)
y_pred_gen_pipe_train = gen_pipe.predict(X_train)
rmse_gen_pipe = root_mean_squared_error(y_test, y_pred_gen_pipe)
r2_gen_pipe = r2_score(y_test, y_pred_gen_pipe)
rmse_gap_gen_pipe =rmse_gen_pipe - root_mean_squared_error(y_train, y_pred_gen_pipe_train)

# printing results
print(f"Metrics for general pipeline:")
print(f"\nRMSE: {rmse_gen_pipe}")
print(f"\nR^2 Score: {r2_gen_pipe}")
print(f"\nRMSE Gap: {rmse_gap_gen_pipe}")






Metrics for general pipeline:

RMSE: 25652.99226352325

R^2 Score: 0.9142049510937665

RMSE Gap: 19477.27670535747


At first glance, these results indicate very strong predictive accuracy, even outperforming the manual approach. However, they also suggest limited generalization ability. A more detailed comparison and evaluation will follow at the end of this chapter.

### 11.2 Insight-Driven Pipeline

In this step, a pipeline is constructed that incorporates insights gained during the manual workflow. This includes strategically handling missing values, applying meaningful ordinal encodings, and integrating engineered features.

At the same time, the structured pipeline framework is maintained to ensure consistent preprocessing and full reproducibility of all transformations.

This approach combines domain understanding with technical best practices, aiming to leverage the strengths of both perspectives.

In [35]:
# building insight-driven pipeline
# building "blind" general pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import root_mean_squared_error, r2_score

# loading original dataframe
df_orig = pd.read_csv("train_housing_prices.csv") 

# preparing dataset
X = df_orig.drop(columns="SalePrice")
y = df_orig["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# identify nominal, ordinal and numerical features
nom_features = [
    "MSZoning", "Neighborhood", "Condition1", "Condition2",
    "BldgType", "HouseStyle", "RoofStyle", "RoofMatl",
    "Exterior1st", "Exterior2nd", "Foundation",
    "Heating", "CentralAir", "Electrical",
    "GarageType", "PavedDrive",
    "LotConfig", "LandContour", "Utilities", "LandSlope",
    "Street", "Alley", "SaleType", "SaleCondition",
    "LotShape", "MasVnrType", "MSSubClass"
]
ord_features = [col for col in X.select_dtypes("object").columns if col not in nom_features]
num_features = [col for col in X.columns if col not in nom_features + ord_features]

# building transformers
from sklearn.base import BaseEstimator, TransformerMixin

## dropping columns: "PoolQC", "Id", "MiscFeature", "MiscVal"
class ColumnDropper(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        X = X.drop(columns=["PoolQC", "Id", "MiscFeature", "MiscVal"])
        return X


## imputing strategic missing values by building StrategicImputer
### impute missing values 
        # in column "Alley" with "NoA", 
        # in colum "MasVnrType" with "NoV" and in column "MasVnrArea" with 0
        # in columns "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1" and "BsmtFinType2" with "NoB"
        # in column "FireplaceQu" with "NoF"
        # in columns "GarageType", "GarageFinish", "GarageQual", "GarageCond" with "NoG" and "GarageYrBlt" with 0
            # add column "HasGarage" to provide missunderstandings regarding year built = 0.
        # in column "Fence" with "NoF"

class StrategicImputer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return(self)
    
    def transform(self, X):
        X = X.copy()
        X["Alley"] = X["Alley"].fillna("NoA")
        X["MasVnrType"] = X["MasVnrType"].fillna("NoV")
        X["MasVnrArea"] = X["MasVnrArea"].fillna(0)
        X["BsmtQual"] = X["BsmtQual"].fillna("NoB")
        X["BsmtCond"] = X["BsmtCond"].fillna("NoB")
        X["BsmtExposure"] = X["BsmtExposure"].fillna("NoB")
        X["BsmtFinType1"] = X["BsmtFinType1"].fillna("NoB")
        X["BsmtFinType2"] = X["BsmtFinType2"].fillna("NoB")
        X["FireplaceQu"] = X["FireplaceQu"].fillna("NoF")
        X["GarageType"] = X["GarageType"].fillna("NoG")
        X["GarageFinish"] = X["GarageFinish"].fillna("NoG")
        X["GarageQual"] = X["GarageQual"].fillna("NoG")
        X["GarageCond"] = X["GarageCond"].fillna("NoG")
        X["GarageYrBlt"] = X["GarageYrBlt"].fillna(0)
        X["HasGarage"] = (X["GarageCars"] > 0).astype(int)
        X["Fence"] = X["Fence"].fillna("NoF")
        return X
        

## mapping ordinal encoding
class OrdinalMapper(BaseEstimator, TransformerMixin):

    ordinal_mapping = {
        "ExterQual": {"Fa": 0, "TA": 1, "Gd": 2, "Ex": 3},
        "ExterCond": {"Po":0, "Fa": 1, "TA": 2, "Gd": 3, "Ex": 4},
        "BsmtQual": {"NoB": 0, "Fa": 1, "TA": 2, "Gd": 3, "Ex": 4},
        "BsmtCond": {"Po":0, "NoB": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
        "HeatingQC": {"Po": 0, "Fa": 1, "TA": 2, "Gd": 3, "Ex": 4},
        "KitchenQual": {"Fa": 0, "TA": 1, "Gd": 2, "Ex": 3},
        "FireplaceQu": {"Po": 0, "NoF": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
        "GarageQual": {"Po":0,"NoG": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
        "GarageCond": {"Po": 0,"NoG": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
        "GarageFinish": {"NoG": 0, "Unf": 1, "RFn": 2, "Fin": 3},
        "BsmtExposure": {"NoB": 0, "No": 1, "Mn": 2, "Av": 3, "Gd": 4},
        "BsmtFinType1": {"NoB": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6},
        "BsmtFinType2": {"NoB": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6},
        "Fence": {"NoF": 0, "MnWw": 1, "GdWo": 2, "MnPrv": 3, "GdPrv": 4},
        "Functional": {"Sal": 0, "Sev": 1, "Maj2": 2, "Maj1": 3, "Mod": 4, "Min2": 5, "Min1": 6, "Typ": 7}
    }

    def fit(self, X, y=None):
        return self

    def transform (self, X):
        X = X.copy()
        for col in self.ordinal_mapping.keys():
            X[col] = X[col].map(self.ordinal_mapping[col])
        return X


## constructing meaningful features: "house_age", "recently_remodeled_or_built", "total_living_area", "total_bathrooms"

class FeatureConstructor(BaseEstimator, TransformerMixin):
    
    def fit (self, X, y=None):
        return self
    
    def transform (self, X):
        X = X.copy()
        X["house_age"] = X["YrSold"] - X["YearBuilt"]
        X["recently_remodeled_or_built"] = (((X["YearRemodAdd"] >= X["YrSold"] - 15) | ((X["YearBuilt"] >= X["YrSold"] - 15))).astype(int))
        X["total_living_area"] = X["TotalBsmtSF"] + X["GrLivArea"]
        X["total_bathrooms"] = X["BsmtFullBath"] + 0.5*X["BsmtHalfBath"] + X["FullBath"] + 0.5*X["HalfBath"]
        return X

# no transformers needed for: 
## imputing statistical missing values: "Electrical -> most frequent"
## encoding nominal features with one-hot encoding  
#  for this tasks, a pipleine in the general preprocess ColumnTransformer is enough

gen_preprocess = ColumnTransformer([
    ("nom_pipe", Pipeline([
        ("stat_imputer", SimpleImputer(strategy="most_frequent")),
        ("nom_encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), nom_features)
], remainder="passthrough")

# for ensurance a transformer, which handles undetected numerical missing values (ordinal values are already numerical too)
class NumericalRestImputer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.medians_ = X.select_dtypes(include="number").median()
        return self 

    def transform(self, X):
        X = X.copy()
        num_col = X.select_dtypes(include="number").columns
        X[num_col] = X[num_col].fillna(self.medians_)
        return X

# modeling (pipeline) 
insight_pipe = Pipeline([
    ("col_drop", ColumnDropper()),
    ("strat_impute", StrategicImputer()),
    ("feat_construct", FeatureConstructor()),
    ("ord_map", OrdinalMapper()),
    ("num_rest_impute", NumericalRestImputer()),
    ("gen_pre", gen_preprocess),
    ("gbr_model", GradientBoostingRegressor())
])

# hyperparameter tuning
param_grid = {
    "gbr_model__n_estimators": [100, 200],
    "gbr_model__max_depth": [3, 5, 10],
    "gbr_model__learning_rate": [0.1, 0.05, 0.01],
    "gbr_model__subsample": [0.8, 1.0]
}

insight_pipe = GridSearchCV(estimator=insight_pipe, param_grid=param_grid, cv=5, n_jobs=-1)
insight_pipe.fit(X_train,y_train)

insight_pipe = insight_pipe.best_estimator_
insight_pipe_pred = insight_pipe.predict(X_test)


# evaluate
rmse = root_mean_squared_error(y_test, insight_pipe_pred)
r2 = r2_score(y_test, insight_pipe_pred)
rmse_gap = rmse - root_mean_squared_error(y_train, insight_pipe.predict(X_train))

print("Metrics for insight-driven Pipeline:")
print(f"\n RMSE:  {rmse}")
print(f"\nR^2 Score:  {r2}")
print(f"\nRMSE Gap:  {rmse_gap}")



Metrics for insight-driven Pipeline:

 RMSE:  26526.38253102154

R^2 Score:  0.9082634882565017

RMSE Gap:  16751.400974454868


### 11.3 Comparison of Modeling Approaches

The insight-driven pipeline seems to be less performative than the general pipeline. To further evaluate the metrics of the different apporaches, we will do a cross validation:

In [36]:
# cross-validation general pipeline and insight-driven pipeline
from sklearn.model_selection import cross_val_score

gen_pipe_cv = -cross_val_score(gen_pipe, X, y, cv=5, scoring="neg_root_mean_squared_error")
insight_pipe_cv = -cross_val_score(insight_pipe, X, y, cv=5, scoring="neg_root_mean_squared_error")

# Comparison manual approach, baseline pipeline and optimized pipeline after cross-validation
cv_comparison = pd.DataFrame({
    "manual": cv_scores_gbr_tuned,
    "baseline pipeline": gen_pipe_cv,
    "optimized pipeline": insight_pipe_cv
})


print("Cross-Validation RMSE Scores for Manual Approach, General Pipeline and Insight-Driven Pipeline:")
print(cv_comparison)
print("\nCross-Validation statistics for Manual Approach, General Pipeline and Optimized Pipeline:")
print(cv_comparison.describe())




Cross-Validation RMSE Scores for Manual Approach, General Pipeline and Insight-Driven Pipeline:
         manual  baseline pipeline  optimized pipeline
0  23298.567461       23223.955440        21302.400900
1  34836.952784       32836.376573        29688.884169
2  25600.502745       27530.206858        25600.157668
3  21920.563200       21268.957175        20955.969052
4  25851.529546       29598.343227        26402.969002

Cross-Validation statistics for Manual Approach, General Pipeline and Optimized Pipeline:
             manual  baseline pipeline  optimized pipeline
count      5.000000           5.000000            5.000000
mean   26301.623147       26891.567855        24790.076158
std     5043.759406        4694.155277         3678.322692
min    21920.563200       21268.957175        20955.969052
25%    23298.567461       23223.955440        21302.400900
50%    25600.502745       27530.206858        25600.157668
75%    25851.529546       29598.343227        26402.969002
max    3483

Despite the initial impression, the cross-validation results reveal that the insight-driven pipeline achieves the highest predictive accuracy, whereas the general pipeline demonstrates slightly better generalization performance. While both pipelines outperform the manual approach, the gap in predictive accuracy between the manual approach and the general pipeline is comparatively small.

## 12. Conclusion

In this project, we analysed three approaches to house price prediction: a manual approach with in-depth data understanding, a purely technical “blind” pipeline, and a combined approach that integrates data-driven insights into a structured pipeline.

In the manual approach, the data was prepared through targeted handling of missing values, feature encoding, and feature selection. Based on this, four models were trained and evaluated, with the Gradient Boosting Regressor achieving the best performance due to its ability to capture non-linear relationships and feature interactions.

Beyond predictive performance, the analysis provided valuable insights into the key drivers of house prices. Living area emerged as the most influential feature, followed by overall quality, neighborhood, and additional quality-related characteristics such as basement condition.

The manual approach already illustrates the fundamental trade-off between predictive accuracy and interpretability, and shows how combining both perspectives leads to a more comprehensive understanding of the problem.

In the purely technical approach, this trade-off shifts: predictive accuracy improves, but interpretability is significantly reduced. Without prior data understanding, it remains unclear which features drive the predictions. Given current societal concerns regarding automated decision-making, this lack of transparency can reduce trust in such models.

The combination of both approaches, however, is not merely a compromise, but a clear enhancement. By integrating data-driven insights into a structured pipeline, preprocessing becomes more effective, leading to improved model performance. This results in the highest predictive accuracy among all three approaches.

While this approach requires additional effort in the preprocessing phase—comparable to the manual approach—the benefits clearly outweigh the costs: improved predictive performance, enhanced interpretability, and a robust, reproducible workflow.

Overall, this project demonstrates that technical workflows do not replace the need for careful, manual data understanding. However, when combined, they lead to significantly stronger results than either approach alone.
